# Integrated Star Workflow (Local PHOENIX)

This notebook variant avoids PHOENIX online download by loading a local saved PHOENIX spectrum (`.npz`) and running SOAP from those arrays.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from create_spectrum_SOAP_local import (
    load_phoenix_from_npz,
    create_spectrum_soap_from_saved_phoenix,
)


In [ ]:
def save_csv_spectrum(path, wave, flux, flux_err=0.001):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    spectrum_df = pd.DataFrame(
        {
            "wave_val": np.asarray(wave, dtype=float),
            "flux_val": np.asarray(flux, dtype=float),
            "flux_err": np.full(len(wave), float(flux_err), dtype=float),
        }
    )
    spectrum_df.to_csv(path, index=False)
    return spectrum_df


def run_local_phoenix_workflow(
    phoenix_npz_path="./saved_spectra/phoenix_output.npz",
    save_dir="./saved_spectra",
    save_csv=True,
    soap_csv_name="soap_local.csv",
):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    wave_phoenix, flux_phoenix = load_phoenix_from_npz(
        npz_path=phoenix_npz_path,
        wave_key="wave_phoenix",
        flux_key="flux_phoenix",
    )

    soap_result = create_spectrum_soap_from_saved_phoenix(
        phoenix_npz_path=phoenix_npz_path,
        teff=5777.0,
        logg=4.44,
        z=0.0,
        psi=[0],
        grid=300,
        inst_reso=115000,
        skip_bis=True,
        skip_fwhm=True,
        skip_rv=False,
        verbose=False,
    )

    # Save SOAP outputs created from local PHOENIX arrays
    np.savez(
        save_dir / "soap_output_local.npz",
        wave_soap=np.asarray(soap_result.wave),
        flux_soap=np.asarray(soap_result.flux),
        ccf_flux_soap=np.asarray(soap_result.ccf_flux),
        rv_ccf_soap=np.asarray(soap_result.rv_ccf),
        spectrum_soap_wave=np.asarray(soap_result.input_spectrum.wave),
        spectrum_soap_flux=np.asarray(soap_result.input_spectrum.flux),
    )

    soap_csv_path = None
    if save_csv:
        soap_csv_path = save_dir / "csv_spectra" / soap_csv_name
        save_csv_spectrum(soap_csv_path, soap_result.wave, soap_result.flux[0])

    return {
        "wave_phoenix": wave_phoenix,
        "flux_phoenix": flux_phoenix,
        "soap_result": soap_result,
        "soap_npz": save_dir / "soap_output_local.npz",
        "soap_csv": soap_csv_path,
    }


In [ ]:
# Set run_local=False if you only want to keep definitions without executing SOAP now.
run_local = False

if run_local:
    local_results = run_local_phoenix_workflow(
        phoenix_npz_path="./saved_spectra/phoenix_output.npz",
        save_dir="./saved_spectra",
        save_csv=True,
        soap_csv_name="soap_local.csv",
    )
    print("Saved:")
    print(" -", local_results["soap_npz"])
    print(" -", local_results["soap_csv"])
else:
    print("run_local is False. Set it to True to execute the local PHOENIX workflow.")


In [ ]:
# Optional quick inspection of local SOAP NPZ contents
soap_npz = Path("./saved_spectra/soap_output_local.npz")
if soap_npz.exists():
    data = np.load(soap_npz)
    print("Available keys:", list(data.files))
    for k in data.files:
        arr = np.asarray(data[k])
        print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")
else:
    print("No local SOAP NPZ found yet:", soap_npz)
